In [ ]:
"""
dataset_generation_synthetic.ipynb
Generates two self-authored synthetic causal-discovery benchmarks with
exact ground truth, saved into this notebook's own directory
(cg_evaluations/datasets/synthetic/).

  1. Synthetic IID        -- linear non-Gaussian SCM (Zheng 2018; Shimizu 2006)
  2. Synthetic TimeSeries -- TTCD Dataset-1 (Faruque et al. 2026, App. C)

See dataset_generation_benchmark.ipynb (../benchmark/) for the two
externally-sourced benchmark datasets (Sachs, nonlinear+confounder), and
dataset_generation.txt (one level up) for the full citation/resource
writeup covering all four datasets.
"""
import os
import json
import numpy as np
import pandas as pd
import networkx as nx

# This notebook lives directly in datasets/synthetic/, so its outputs are
# saved right alongside it.
DATASETS_DIR = "."
os.makedirs(DATASETS_DIR, exist_ok=True)
print(f"Saving into: {os.path.abspath(DATASETS_DIR)}")

In [ ]:
def generate_synthetic_iid(
    d: int = 8,
    n_rows: int = 1000,
    edge_density: float = 1.8,     # expected edges approximately edge_density * d
    noise: str = "exponential",    # 'exponential' | 'gumbel' | 'gaussian'
    weight_ranges=((-2.0, -0.5), (0.5, 2.0)),
    seed: int = 42,
):
    """
    Linear SEM X = W^T X + eps over a random Erdos-Renyi DAG.

    The graph and SEM generation follow the benchmark procedures used by
    Zheng et al. (2018, NOTEARS) and Yu et al. (2019, DAG-GNN).

    Non-Gaussian exponential noise satisfies the identifiability assumption
    used by LiNGAM (Shimizu et al., 2006). Other methods are evaluated on the
    same generated dataset for a consistent comparison.

    Returns
    -------
    df : pandas.DataFrame
        Generated dataset with shape (n_rows, d).

    true_edges : list of dict
        Ground-truth edges in the form:
        [{'cause': 'X1', 'effect': 'X2'}, ...]

    meta : dict
        Dataset settings and verification results.
    """

    if d < 2:
        raise ValueError("d must be at least 2.")

    if n_rows < 1:
        raise ValueError("n_rows must be positive.")

    if edge_density < 0:
        raise ValueError("edge_density must be non-negative.")

    if noise not in {"exponential", "gumbel", "gaussian"}:
        raise ValueError(
            f"unknown noise {noise!r}; choose "
            "'exponential', 'gumbel', or 'gaussian'"
        )

    rng = np.random.default_rng(seed)
    cols = [f"X{i+1}" for i in range(d)]

    # Step 1: random DAG, acyclic by construction
    # Upper-triangular edges under a random topological ordering.
    max_possible_edges = d * (d - 1) / 2
    expected_edges = edge_density * d
    p_edge = min(1.0, expected_edges / max_possible_edges)

    perm = rng.permutation(d)
    mask = np.zeros((d, d), dtype=bool)

    for a in range(d):
        for b in range(a + 1, d):
            if rng.random() < p_edge:
                mask[perm[a], perm[b]] = True

    # Create graph once and verify acyclicity.
    G = nx.DiGraph()
    G.add_nodes_from(range(d))
    G.add_edges_from(
        (i, j)
        for i in range(d)
        for j in range(d)
        if mask[i, j]
    )

    if not nx.is_directed_acyclic_graph(G):
        raise RuntimeError("Generated graph is unexpectedly cyclic.")

    # Step 2: edge weights bounded away from zero
    W = np.zeros((d, d), dtype=float)

    for i in range(d):
        for j in range(d):
            if mask[i, j]:
                range_idx = rng.integers(0, len(weight_ranges))
                lo, hi = weight_ranges[range_idx]

                if lo >= hi:
                    raise ValueError(
                        f"Invalid weight range ({lo}, {hi}): "
                        "lower bound must be smaller than upper bound."
                    )

                if lo <= 0 <= hi:
                    raise ValueError(
                        f"Weight range ({lo}, {hi}) contains zero. "
                        "Use ranges bounded away from zero."
                    )

                W[i, j] = rng.uniform(lo, hi)

    # Step 3: generate data in topological order
    def draw_noise(size):
        if noise == "exponential":
            # Exponential(scale=1) has theoretical mean 1.
            # Subtracting 1 preserves independence across samples.
            return rng.exponential(scale=1.0, size=size) - 1.0

        if noise == "gumbel":
            # Standard Gumbel mean is Euler's constant.
            # np.euler_gamma may not exist in older NumPy versions,
            # so use the numerical constant directly.
            euler_gamma = 0.5772156649015329
            return rng.gumbel(loc=0.0, scale=1.0, size=size) - euler_gamma

        if noise == "gaussian":
            return rng.normal(loc=0.0, scale=1.0, size=size)

        raise ValueError(f"unknown noise {noise!r}")

    X = np.zeros((n_rows, d), dtype=float)

    for j in nx.topological_sort(G):
        val = draw_noise(n_rows)

        for i in np.where(mask[:, j])[0]:
            val = val + W[i, j] * X[:, i]

        X[:, j] = val

    df = pd.DataFrame(X, columns=cols)

    true_edges = [
        {"cause": cols[i], "effect": cols[j]}
        for i in range(d)
        for j in range(d)
        if mask[i, j]
    ]

    meta = {
        "name": "synthetic_iid",
        "kind": "iid",
        "d": d,
        "n_rows": n_rows,
        "noise": noise,
        "seed": seed,
        "columns": cols,
        "n_edges": int(mask.sum()),
        "edge_density": edge_density,
        "edge_probability": float(p_edge),
        "expected_edges": float(expected_edges),
        "reference": (
            "Zheng et al. 2018 (NOTEARS); "
            "Yu et al. 2019 (DAG-GNN); "
            "non-Gaussian noise per Shimizu et al. 2006 (LiNGAM)"
        ),
    }

    # Verification: count unshielded colliders Xi -> Xj <- Xk
    colliders = 0

    for j in range(d):
        pars = np.where(mask[:, j])[0]

        for a in range(len(pars)):
            for b in range(a + 1, len(pars)):
                parent_a = pars[a]
                parent_b = pars[b]

                if not (
                    mask[parent_a, parent_b]
                    or mask[parent_b, parent_a]
                ):
                    colliders += 1

    meta["verification"] = {
        "acyclic": nx.is_directed_acyclic_graph(G),
        "no_constant_column": bool(
            (df.std().to_numpy() > 1e-8).all()
        ),
        "no_nan": bool(not df.isna().any().any()),
        "finite_values": bool(np.isfinite(X).all()),
        "n_v_structures": int(colliders),
        "rows_ok": len(df) >= 30,
    }

    meta["verification"]["pass"] = all([
        meta["verification"]["acyclic"],
        meta["verification"]["no_constant_column"],
        meta["verification"]["no_nan"],
        meta["verification"]["finite_values"],
        colliders >= 1,
        int(mask.sum()) >= 2,
        len(df) >= 30,
    ])

    return df, true_edges, meta

In [ ]:
def generate_synthetic_timeseries(T: int = 3000, burn_in: int = 200, seed: int = 42):
    """
    TTCD Dataset-1 verbatim (arXiv:2605.08111, Appendix C). Four variables,
    Gaussian noise, sinusoidal NON-STATIONARITY, lagged + contemporaneous
    links; X1 is a common cause of all others.
    Returns (df, true_edges, meta). true_edges: [{'cause','effect','lag'}]
    """
    rng = np.random.default_rng(seed)
    total = T + burn_in
    X1 = np.zeros(total); X2 = np.zeros(total)
    X3 = np.zeros(total); X4 = np.zeros(total)
    e1 = rng.normal(0, 1, total); e2 = rng.normal(0, 1, total)
    e3 = rng.normal(0, 1, total); e4 = rng.normal(0, 1, total)

    for t in range(total):
        x1_5 = X1[t-5] if t >= 5 else 0.0
        x1_2 = X1[t-2] if t >= 2 else 0.0
        x1_1 = X1[t-1] if t >= 1 else 0.0
        x3_1 = X3[t-1] if t >= 1 else 0.0
        x4_1 = X4[t-1] if t >= 1 else 0.0
        X1[t] = 0.5*x1_5 + 0.5*x1_2 + e1[t]
        X2[t] = 0.1*X1[t] + 0.7*x1_1 + 1.5*np.sin(t/50) + e2[t]
        X3[t] = 0.8*x1_1 + e3[t]
        X4[t] = (0.2*x4_1 + 0.4*X3[t] + 0.4*x3_1 + 0.4*x1_1
                 + np.sin(t/50) + np.sin(t/20) + e4[t])

    data = np.column_stack([X1, X2, X3, X4])[burn_in:]
    df = pd.DataFrame(data, columns=["X1", "X2", "X3", "X4"])

    true_edges = [
        {"cause": "X1", "effect": "X1", "lag": 5},
        {"cause": "X1", "effect": "X1", "lag": 2},
        {"cause": "X1", "effect": "X2", "lag": 0},
        {"cause": "X1", "effect": "X2", "lag": 1},
        {"cause": "X1", "effect": "X3", "lag": 1},
        {"cause": "X3", "effect": "X4", "lag": 0},
        {"cause": "X3", "effect": "X4", "lag": 1},
        {"cause": "X1", "effect": "X4", "lag": 1},
        {"cause": "X4", "effect": "X4", "lag": 1},
    ]
    meta = {
        "name": "synthetic_timeseries_ttcd_dataset1", "kind": "timeseries",
        "T": T, "seed": seed, "max_lag": 5, "columns": ["X1", "X2", "X3", "X4"],
        "reference": "Faruque, Ali, Zheng, Wang 2026 (TTCD), arXiv:2605.08111, "
                     "Appendix C, Dataset-1",
        "note": "Non-stationary; PCMCI/PCMCI+ assume stationarity, so lower "
                "scores here are an expected benchmark property.",
        "verification": {
            "no_constant_column": bool((df.std().to_numpy() > 1e-8).all()),
            "no_nan": bool(not df.isna().any().any()),
            "rows_ok": len(df) >= 30,
        },
    }
    meta["verification"]["pass"] = all([
        meta["verification"]["no_constant_column"],
        meta["verification"]["no_nan"], meta["verification"]["rows_ok"],
    ])
    return df, true_edges, meta

In [ ]:
def save_dataset(name, df, true_edges, meta):
    """Saves one dataset's three files into DATASETS_DIR, in the format
    the evaluation notebooks (evaluate_cg.ipynb / evaluate_cg_bench.ipynb)
    expect via their own load_dataset_and_truth() helper:
       {name}.csv            -- the data
       {name}_truth.json     -- ground-truth edges, used for scoring
       {name}_meta.json      -- parameters, citations, verification report
    """
    df.to_csv(f"{DATASETS_DIR}/{name}.csv", index=False)
    with open(f"{DATASETS_DIR}/{name}_truth.json", "w") as f:
        json.dump(true_edges, f, indent=2)
    meta_serializable = {k: v for k, v in meta.items()}  # meta has no numpy here
    with open(f"{DATASETS_DIR}/{name}_meta.json", "w") as f:
        json.dump(meta_serializable, f, indent=2)
    status = "PASS" if meta["verification"]["pass"] else "FAIL"
    print(f"[{status}] {name}: {df.shape[0]}x{df.shape[1]}, "
          f"{len(true_edges)} true edges  -> saved 3 files")

# --- IID ---
iid_df, iid_edges, iid_meta = generate_synthetic_iid()
save_dataset("synthetic_iid", iid_df, iid_edges, iid_meta)
print("  verification:", iid_meta["verification"])
print("  true edges:", [(e['cause'], e['effect']) for e in iid_edges])

print()

# --- Time series ---
ts_df, ts_edges, ts_meta = generate_synthetic_timeseries()
save_dataset("synthetic_timeseries", ts_df, ts_edges, ts_meta)
print("  verification:", ts_meta["verification"])
print("  true edges:", [(e['cause'], e['effect'], e['lag']) for e in ts_edges])

In [ ]:
# One-off sanity check for the generator itself, not part of the regular
# dataset-generation flow above: running PC at a large sample size should
# recover most of the planted IID skeleton if the generator is producing a
# genuinely learnable structure. Requires causal-learn (already a project
# dependency).
def sanity_check_iid(seed=42, n_rows=5000):
    from causallearn.search.ConstraintBased.PC import pc as cl_pc
    df, true_edges, meta = generate_synthetic_iid(n_rows=n_rows, seed=seed)
    cols = meta["columns"]
    cg = cl_pc(df.to_numpy(), alpha=0.01, node_names=cols, show_progress=False)
    g = cg.G.graph
    true_skel = {frozenset((e["cause"], e["effect"])) for e in true_edges}
    pred_skel = {frozenset((cols[i], cols[j]))
                 for i in range(len(cols)) for j in range(i+1, len(cols))
                 if g[i, j] != 0 or g[j, i] != 0}
    tp = len(true_skel & pred_skel)
    verdict = "OK — generator sound" if tp >= 0.8*len(true_skel) else "CHECK generator"
    print(f"PC @N={n_rows}: recovered {tp}/{len(true_skel)} true skeleton edges, "
          f"{len(pred_skel - true_skel)} extra.  {verdict}")

sanity_check_iid()